# Knowledge Distillation: Teaching Small Models to Mimic Large Ones

**Knowledge distillation** is one of the most practical techniques for deploying models in production. It teaches small, fast models to mimic large, accurate models - achieving 10-100x speedup with minimal accuracy loss.

## What You'll Learn

This notebook builds deep intuitions about knowledge distillation through hands-on implementation:

1. **Teacher-student framework** - how knowledge transfer works
2. **Soft targets** - why they're more informative than hard labels
3. **Temperature scaling** - controlling the "softness" of predictions
4. **Response-based distillation** - learning from output probabilities
5. **Feature-based distillation** - learning from intermediate representations
6. **Practical deployment** - model compression, speedup, and trade-offs

## Why Knowledge Distillation Matters

Large models are accurate but impractical for deployment:
- **Slow inference**: 100ms+ per prediction (unusable for real-time)
- **Large memory**: GB of RAM (won't fit on mobile/edge devices)
- **High cost**: Expensive GPUs required

Knowledge distillation enables:
- **10-100x speedup**: Sub-millisecond inference
- **10-100x smaller models**: MB instead of GB
- **Minimal accuracy loss**: Often <2% drop
- **CPU/mobile deployment**: Run anywhere

## The Key Insight

**Hard labels** (one-hot: [0, 1, 0, 0]) lose information:
- Only tell you the correct class
- Don't reveal relationships between classes

**Soft targets** from a teacher model (e.g., [0.02, 0.85, 0.10, 0.03]) are much richer:
- Show **relative similarities** between classes
- Encode **dark knowledge** about class relationships
- Provide **smoother gradients** for training

Example: A cat image might be 85% cat, 10% dog, 5% other. This tells the student: "cats are more similar to dogs than to cars!"

## Setup

Let's import the necessary libraries and set up our environment.

In [ ]:
# Standard library imports

# Third-party imports

# Local library imports
    get_device, set_seed, count_parameters,
    create_dataset, create_dataloaders,
    get_dataset_config,
)

# Enable autoreload for hot reloading of library changes
%load_ext autoreload
%autoreload 2

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

### Set Random Seeds and Device

For reproducible results, we set random seeds.

In [ ]:
from aiml_notebooks import (

In [ ]:
# Set random seed for reproducibility
set_seed(42)

# Configure device
device = get_device()
print(f"Using device: {device}")

## Part 1: Load and Prepare Dataset

We'll use **CIFAR-10** to demonstrate knowledge distillation. This is a realistic scenario - we want to deploy image classification but need models small enough for mobile devices.

In [ ]:
# Get CIFAR-10 configuration
dataset_config = get_dataset_config('cifar10')
class_names = dataset_config['classes']
num_classes = len(class_names)

print(f"Dataset: CIFAR-10")
print(f"Number of classes: {num_classes}")
print(f"Classes: {', '.join(class_names)}")

### Create DataLoaders

We'll load CIFAR-10 with standard data augmentation for training.

In [ ]:
# Load CIFAR-10 dataset
train_dataset, test_dataset = create_dataset(dataset_id='cifar10')

# Create data loaders
BATCH_SIZE = 128
train_loader, test_loader = create_dataloaders(
    train_dataset=train_dataset,
    val_dataset=test_dataset,
    batch_size=BATCH_SIZE,
    num_workers=2,
    use_collate_fn=False
)

print(f"\nDataset sizes:")
print(f"  Train: {len(train_dataset):,} samples")
print(f"  Test: {len(test_dataset):,} samples")
print(f"\nDataLoader info:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")

## Part 2: Define Teacher and Student Architectures

### The Teacher-Student Framework

**Teacher model**: Large, accurate, but slow
- High capacity (many parameters)
- Trained on hard labels to achieve best accuracy
- Used only during training (offline)

**Student model**: Small, fast, but less capacity
- Low capacity (few parameters)
- Learns from both hard labels AND teacher's soft predictions
- Deployed for inference (production)

The student model tries to **mimic the teacher's behavior**, not just match the labels.

### Define Teacher Model (Large ResNet)

We'll use a deep ResNet with many filters as our teacher.

In [ ]:
class ResidualBlock(nn.Module):
    """Residual block with skip connection."""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Skip connection with projection if dimensions change
        self.skip = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, 
                         stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        identity = self.skip(x)
        
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        out = F.relu(out)
        return out


class TeacherModel(nn.Module):
    """Large teacher model with high capacity."""
    def __init__(self, num_classes=10):
        super().__init__()
        # Initial convolution
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        
        # Residual blocks (deep network)
        self.layer1 = self._make_layer(64, 64, 3)
        self.layer2 = self._make_layer(64, 128, 3, stride=2)
        self.layer3 = self._make_layer(128, 256, 3, stride=2)
        self.layer4 = self._make_layer(256, 512, 3, stride=2)
        
        # Classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
    
    def _make_layer(self, in_channels, out_channels, num_blocks, stride=1):
        layers = []
        layers.append(ResidualBlock(in_channels, out_channels, stride))
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_channels, out_channels))
        return nn.Sequential(*layers)
    
    def forward(self, x, return_features=False):
        # Initial conv
        x = F.relu(self.bn1(self.conv1(x)))
        
        # Residual blocks
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        # Pool and classify
        features = self.avgpool(x)
        features = features.view(features.size(0), -1)
        logits = self.fc(features)
        
        if return_features:
            return logits, features
        return logits


# Create teacher model
teacher = TeacherModel(num_classes=num_classes).to(device)
teacher_params = count_parameters(teacher)

print("Teacher Model (Large ResNet):")
print(f"  Parameters: {teacher_params:,}")
print(f"  Architecture: Deep (12 residual blocks) with wide layers (512 channels)")

### Define Student Model (Small CNN)

Our student is a simple, shallow CNN - much smaller and faster than the teacher.

In [ ]:
class StudentModel(nn.Module):
    """Small student model with low capacity."""
    def __init__(self, num_classes=10):
        super().__init__()
        # Simple CNN (shallow and narrow)
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)
    
    def forward(self, x, return_features=False):
        # Conv layers
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # 32x32 -> 16x16
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # 16x16 -> 8x8
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # 8x8 -> 4x4
        
        # Global pooling and classify
        features = self.avgpool(x)
        features = features.view(features.size(0), -1)
        logits = self.fc(features)
        
        if return_features:
            return logits, features
        return logits


# Create student model
student = StudentModel(num_classes=num_classes).to(device)
student_params = count_parameters(student)

print("\nStudent Model (Small CNN):")
print(f"  Parameters: {student_params:,}")
print(f"  Architecture: Shallow (3 conv layers) with narrow layers (128 channels)")
print(f"\nCompression ratio: {teacher_params / student_params:.1f}x smaller")
print(f"Student has only {student_params / teacher_params * 100:.1f}% of teacher's parameters!")

## Part 3: Train the Teacher Model

First, we need a strong teacher. We'll train the large model on hard labels to achieve the best possible accuracy.

### Define Training and Evaluation Functions

In [ ]:
from tqdm.auto import tqdm

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, desc="Training"):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in tqdm(loader, desc=desc, leave=False):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), 100. * correct / total


def evaluate(model, loader, criterion, device, desc="Evaluating"):
    """Evaluate the model."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc=desc, leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), 100. * correct / total

print("✓ Training functions defined")

### Train the Teacher

We'll train the teacher model using standard supervised learning with cross-entropy loss.

In [ ]:
# Training configuration
NUM_EPOCHS = 20
criterion = nn.CrossEntropyLoss()
teacher_optimizer = optim.Adam(teacher.parameters(), lr=1e-3)

print("Training Teacher Model...\n")
teacher_history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(teacher, train_loader, criterion, 
                                       teacher_optimizer, device, "Teacher Train")
    test_loss, test_acc = evaluate(teacher, test_loader, criterion, device, "Teacher Eval")
    
    teacher_history['train_loss'].append(train_loss)
    teacher_history['train_acc'].append(train_acc)
    teacher_history['test_loss'].append(test_loss)
    teacher_history['test_acc'].append(test_acc)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}: Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

print(f"\n✓ Teacher training complete!")
print(f"Final teacher test accuracy: {teacher_history['test_acc'][-1]:.2f}%")

### Visualize Teacher Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(teacher_history['train_loss'], label='Train', marker='o')
axes[0].plot(teacher_history['test_loss'], label='Test', marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Teacher Model Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(teacher_history['train_acc'], label='Train', marker='o')
axes[1].plot(teacher_history['test_acc'], label='Test', marker='o')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Teacher Model Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"The teacher achieved {teacher_history['test_acc'][-1]:.2f}% accuracy.")
print(f"Now we'll teach a small student model to mimic this performance!")

## Part 4: Baseline - Train Student Without Distillation

Before using knowledge distillation, let's establish a baseline: training the small student model directly on hard labels (same as teacher training, but with a smaller model).

### Train Vanilla Student

This is standard supervised learning - no teacher involved.

In [ ]:
# Create fresh student model for baseline
student_vanilla = StudentModel(num_classes=num_classes).to(device)
student_vanilla_optimizer = optim.Adam(student_vanilla.parameters(), lr=1e-3)

print("Training Vanilla Student (no distillation)...\n")
vanilla_history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(student_vanilla, train_loader, criterion,
                                       student_vanilla_optimizer, device, "Vanilla Train")
    test_loss, test_acc = evaluate(student_vanilla, test_loader, criterion, device, "Vanilla Eval")
    
    vanilla_history['train_loss'].append(train_loss)
    vanilla_history['train_acc'].append(train_acc)
    vanilla_history['test_loss'].append(test_loss)
    vanilla_history['test_acc'].append(test_acc)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}: Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

print(f"\n✓ Vanilla student training complete!")
print(f"Final test accuracy: {vanilla_history['test_acc'][-1]:.2f}%")
print(f"\nPerformance gap:")
print(f"  Teacher: {teacher_history['test_acc'][-1]:.2f}%")
print(f"  Vanilla Student: {vanilla_history['test_acc'][-1]:.2f}%")
print(f"  Gap: {teacher_history['test_acc'][-1] - vanilla_history['test_acc'][-1]:.2f}%")

### Understanding the Performance Gap

The vanilla student typically underperforms the teacher because:
1. **Limited capacity**: Fewer parameters = less expressive power
2. **Hard labels are sparse**: Only one bit of information per sample (correct class)
3. **No class relationships**: The model doesn't learn that "dog" is more similar to "cat" than to "airplane"

Knowledge distillation addresses points 2 and 3 by using the teacher's soft predictions!

## Part 5: Understanding Soft Targets and Temperature

### Hard Labels vs Soft Targets

Let's visualize the difference between hard labels and soft predictions.

In [ ]:
# Get a batch of examples
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

# Get teacher predictions
teacher.eval()
with torch.no_grad():
    teacher_logits = teacher(images)
    teacher_probs = F.softmax(teacher_logits, dim=1)

# Pick one example
idx = 0
true_label = labels[idx].item()
teacher_pred = teacher_probs[idx].cpu().numpy()

# Create hard label (one-hot)
hard_label = np.zeros(num_classes)
hard_label[true_label] = 1.0

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Show image
img = images[idx].cpu().permute(1, 2, 0).numpy()
img = (img - img.min()) / (img.max() - img.min())  # Normalize for display
axes[0].imshow(img)
axes[0].set_title(f'True Label: {class_names[true_label]}')
axes[0].axis('off')

# Hard label
axes[1].bar(range(num_classes), hard_label)
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Probability')
axes[1].set_title('Hard Label (one-hot)')
axes[1].set_xticks(range(num_classes))
axes[1].set_xticklabels([c[:3] for c in class_names], rotation=45)
axes[1].set_ylim([0, 1])

# Soft target
axes[2].bar(range(num_classes), teacher_pred)
axes[2].set_xlabel('Class')
axes[2].set_ylabel('Probability')
axes[2].set_title('Soft Target (teacher predictions)')
axes[2].set_xticks(range(num_classes))
axes[2].set_xticklabels([c[:3] for c in class_names], rotation=45)
axes[2].set_ylim([0, 1])

plt.tight_layout()
plt.show()

print("Hard label: only tells us the correct class")
print("Soft target: reveals relationships between classes!")
print(f"\nTop 3 teacher predictions:")
top3_indices = teacher_pred.argsort()[-3:][::-1]
for i in top3_indices:
    print(f"  {class_names[i]}: {teacher_pred[i]*100:.1f}%")

### Temperature Scaling

**Problem**: Neural network predictions are often very confident (e.g., [0.001, 0.995, 0.001, ...]). This is good for accuracy but bad for distillation - the soft targets become almost as sparse as hard labels!

**Solution**: **Temperature scaling** softens the probability distribution.

**How it works:**
1. Divide logits by temperature $T$ before softmax: $p_i = \frac{e^{z_i/T}}{\sum_j e^{z_j/T}}$
2. Higher $T$ → softer (more uniform) distribution
3. Lower $T$ → sharper (more peaked) distribution
4. $T=1$ → normal softmax

**Intuition**: Temperature controls how "confident" the predictions appear. High temperature reveals more subtle class relationships.

### Visualize Effect of Temperature

In [ ]:
def softmax_with_temperature(logits, temperature):
    """Apply softmax with temperature scaling."""
    return F.softmax(logits / temperature, dim=1)

# Get teacher logits for one example
example_logits = teacher_logits[idx:idx+1]

# Try different temperatures
temperatures = [1, 2, 5, 10]
fig, axes = plt.subplots(1, len(temperatures), figsize=(16, 4))

for i, T in enumerate(temperatures):
    probs = softmax_with_temperature(example_logits, T).cpu().numpy()[0]
    
    axes[i].bar(range(num_classes), probs)
    axes[i].set_xlabel('Class')
    axes[i].set_ylabel('Probability')
    axes[i].set_title(f'Temperature T={T}')
    axes[i].set_xticks(range(num_classes))
    axes[i].set_xticklabels([c[:3] for c in class_names], rotation=45)
    axes[i].set_ylim([0, max(0.5, probs.max() * 1.1)])

plt.tight_layout()
plt.show()

print("Observations:")
print("  T=1: Sharp distribution (confident predictions)")
print("  T=2-5: Softer distribution (reveals class relationships)")
print("  T=10: Very soft (almost uniform - too much!)")
print("\nTypical choice: T=3 to T=5 for distillation")

## Part 6: Response-Based Distillation (Soft Targets)

Now let's implement knowledge distillation! The student learns from:
1. **Hard labels** (ground truth) via cross-entropy
2. **Soft targets** (teacher predictions) via KL divergence

The loss function is:
$$L = \alpha \cdot L_{hard} + (1-\alpha) \cdot T^2 \cdot L_{soft}$$

where:
- $L_{hard}$ = cross-entropy with hard labels
- $L_{soft}$ = KL divergence between student and teacher (both at temperature $T$)
- $\alpha$ = weight balancing hard and soft losses (typically 0.1-0.3)
- $T^2$ = temperature scaling factor (compensates for gradient magnitude)

### Define Distillation Loss

In [ ]:
class DistillationLoss(nn.Module):
    """Knowledge distillation loss combining hard and soft targets."""
    def __init__(self, temperature=3.0, alpha=0.2):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha  # Weight for hard loss
        self.ce_loss = nn.CrossEntropyLoss()
    
    def forward(self, student_logits, teacher_logits, labels):
        # Hard loss: student predictions vs true labels
        hard_loss = self.ce_loss(student_logits, labels)
        
        # Soft loss: student predictions vs teacher predictions (both with temperature)
        student_soft = F.log_softmax(student_logits / self.temperature, dim=1)
        teacher_soft = F.softmax(teacher_logits / self.temperature, dim=1)
        soft_loss = F.kl_div(student_soft, teacher_soft, reduction='batchmean')
        
        # Combined loss (T^2 factor compensates for gradient magnitude)
        loss = self.alpha * hard_loss + (1 - self.alpha) * (self.temperature ** 2) * soft_loss
        
        return loss, hard_loss, soft_loss

print("✓ Distillation loss defined")
print(f"\nLoss components:")
print(f"  1. Hard loss (α={0.2}): Student vs ground truth labels")
print(f"  2. Soft loss (1-α={0.8}): Student vs teacher predictions")
print(f"  3. Temperature: T={3.0} for soft targets")

### Train Student with Distillation

In [ ]:
def train_epoch_distillation(student, teacher, loader, distill_criterion, optimizer, device):
    """Train student with knowledge distillation."""
    student.train()
    teacher.eval()  # Teacher is frozen
    
    total_loss = 0
    total_hard_loss = 0
    total_soft_loss = 0
    correct = 0
    total = 0
    
    for images, labels in tqdm(loader, desc="Distillation Train", leave=False):
        images, labels = images.to(device), labels.to(device)
        
        # Get teacher predictions (no gradient)
        with torch.no_grad():
            teacher_logits = teacher(images)
        
        # Get student predictions
        optimizer.zero_grad()
        student_logits = student(images)
        
        # Compute distillation loss
        loss, hard_loss, soft_loss = distill_criterion(student_logits, teacher_logits, labels)
        
        loss.backward()
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        total_hard_loss += hard_loss.item()
        total_soft_loss += soft_loss.item()
        _, predicted = student_logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    n = len(loader)
    return total_loss/n, total_hard_loss/n, total_soft_loss/n, 100.*correct/total

print("✓ Distillation training function defined")

### Run Distillation Training

Now we train a student model using knowledge distillation.

In [ ]:
# Create fresh student model for distillation
student_distill = StudentModel(num_classes=num_classes).to(device)
student_distill_optimizer = optim.Adam(student_distill.parameters(), lr=1e-3)

# Create distillation loss
distill_criterion = DistillationLoss(temperature=3.0, alpha=0.2)

print("Training Student with Knowledge Distillation...\n")
distill_history = {
    'train_loss': [], 'train_hard_loss': [], 'train_soft_loss': [],
    'train_acc': [], 'test_loss': [], 'test_acc': []
}

for epoch in range(NUM_EPOCHS):
    # Train with distillation
    train_loss, hard_loss, soft_loss, train_acc = train_epoch_distillation(
        student_distill, teacher, train_loader, distill_criterion,
        student_distill_optimizer, device
    )
    
    # Evaluate (standard evaluation)
    test_loss, test_acc = evaluate(student_distill, test_loader, criterion, device, "Distill Eval")
    
    distill_history['train_loss'].append(train_loss)
    distill_history['train_hard_loss'].append(hard_loss)
    distill_history['train_soft_loss'].append(soft_loss)
    distill_history['train_acc'].append(train_acc)
    distill_history['test_loss'].append(test_loss)
    distill_history['test_acc'].append(test_acc)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}: Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

print(f"\n✓ Distillation training complete!")
print(f"Final test accuracy: {distill_history['test_acc'][-1]:.2f}%")

### Compare All Models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
axes[0].plot(teacher_history['test_acc'], label='Teacher (large)', marker='o', linewidth=2)
axes[0].plot(vanilla_history['test_acc'], label='Student (no distillation)', marker='s', linewidth=2)
axes[0].plot(distill_history['test_acc'], label='Student (with distillation)', marker='^', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Model Comparison: Test Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Hard vs soft loss components during distillation
axes[1].plot(distill_history['train_hard_loss'], label='Hard loss (labels)', marker='o')
axes[1].plot(distill_history['train_soft_loss'], label='Soft loss (teacher)', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Distillation Loss Components')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"Teacher (large):               {teacher_history['test_acc'][-1]:.2f}%")
print(f"Student (no distillation):     {vanilla_history['test_acc'][-1]:.2f}%")
print(f"Student (with distillation):   {distill_history['test_acc'][-1]:.2f}%")
print("="*60)
print(f"Distillation improvement: +{distill_history['test_acc'][-1] - vanilla_history['test_acc'][-1]:.2f}%")
print(f"Gap closed: {(distill_history['test_acc'][-1] - vanilla_history['test_acc'][-1]) / (teacher_history['test_acc'][-1] - vanilla_history['test_acc'][-1]) * 100:.1f}%")
print("="*60)

### Key Observations

1. **Distillation improves student performance**: The distilled student typically outperforms the vanilla student by 1-3%

2. **Can't fully match teacher**: The student has limited capacity - it closes part of the gap but not all

3. **Soft loss dominates early**: The student learns general patterns from teacher predictions

4. **Hard loss matters later**: Ground truth labels provide correction signal

5. **Model is much smaller**: Same performance with ~5x fewer parameters!

## Part 7: Feature-Based Distillation

**Limitation of response-based distillation**: Only uses final output probabilities.

**Feature-based distillation** goes further:
- Match **intermediate layer activations** between teacher and student
- Student learns to extract similar features, not just produce similar outputs
- Often gives better results, especially for complex tasks

**Approach**: Add loss terms that minimize distance between student and teacher features at intermediate layers.

### Define Feature Distillation Loss

In [ ]:
class FeatureDistillationLoss(nn.Module):
    """Knowledge distillation with both output and feature matching."""
    def __init__(self, temperature=3.0, alpha=0.2, beta=0.1):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha    # Weight for hard loss
        self.beta = beta      # Weight for feature loss
        self.ce_loss = nn.CrossEntropyLoss()
        self.mse_loss = nn.MSELoss()
    
    def forward(self, student_logits, teacher_logits, student_features, teacher_features, labels):
        # Hard loss: student predictions vs true labels
        hard_loss = self.ce_loss(student_logits, labels)
        
        # Soft loss: student predictions vs teacher predictions
        student_soft = F.log_softmax(student_logits / self.temperature, dim=1)
        teacher_soft = F.softmax(teacher_logits / self.temperature, dim=1)
        soft_loss = F.kl_div(student_soft, teacher_soft, reduction='batchmean')
        
        # Feature loss: match intermediate representations
        # Note: Teacher features may have different dimension, so we need a projection
        feature_loss = self.mse_loss(student_features, teacher_features)
        
        # Combined loss
        loss = (self.alpha * hard_loss + 
                (1 - self.alpha - self.beta) * (self.temperature ** 2) * soft_loss +
                self.beta * feature_loss)
        
        return loss, hard_loss, soft_loss, feature_loss

print("✓ Feature distillation loss defined")
print(f"\nLoss components:")
print(f"  1. Hard loss (α={0.2}): Student vs ground truth")
print(f"  2. Soft loss (1-α-β={0.7}): Student vs teacher outputs")
print(f"  3. Feature loss (β={0.1}): Student vs teacher features")

### Add Feature Projection Layers

Since teacher and student have different feature dimensions (512 vs 128), we need a projection layer to match dimensions for feature distillation.

In [ ]:
class StudentWithProjection(nn.Module):
    """Student model with feature projection for distillation."""
    def __init__(self, base_model, teacher_feature_dim=512):
        super().__init__()
        self.student = base_model
        
        # Get student feature dimension (before final classifier)
        student_feature_dim = base_model.fc.in_features
        
        # Projection layer to match teacher feature dimensions
        self.feature_projection = nn.Linear(student_feature_dim, teacher_feature_dim)
    
    def forward(self, x):
        # Get student features and logits
        logits, student_features = self.student(x, return_features=True)
        
        # Project to teacher dimension
        projected_features = self.feature_projection(student_features)
        
        return logits, projected_features

# Create student with projection
student_feat = StudentModel(num_classes=num_classes).to(device)
student_feat_model = StudentWithProjection(student_feat, teacher_feature_dim=512).to(device)

print("✓ Student with feature projection created")
print(f"  Student features: 128-dim")
print(f"  Projected to: 512-dim (matches teacher)")

### Train with Feature Distillation

In [ ]:
def train_epoch_feature_distillation(student, teacher, loader, distill_criterion, optimizer, device):
    """Train student with feature-based distillation."""
    student.train()
    teacher.eval()
    
    total_loss = 0
    total_hard_loss = 0
    total_soft_loss = 0
    total_feat_loss = 0
    correct = 0
    total = 0
    
    for images, labels in tqdm(loader, desc="Feature Distill", leave=False):
        images, labels = images.to(device), labels.to(device)
        
        # Get teacher predictions and features (no gradient)
        with torch.no_grad():
            teacher_logits, teacher_features = teacher(images, return_features=True)
        
        # Get student predictions and features
        optimizer.zero_grad()
        student_logits, student_features = student(images)
        
        # Compute feature distillation loss
        loss, hard_loss, soft_loss, feat_loss = distill_criterion(
            student_logits, teacher_logits, student_features, teacher_features, labels
        )
        
        loss.backward()
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        total_hard_loss += hard_loss.item()
        total_soft_loss += soft_loss.item()
        total_feat_loss += feat_loss.item()
        _, predicted = student_logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    n = len(loader)
    return total_loss/n, total_hard_loss/n, total_soft_loss/n, total_feat_loss/n, 100.*correct/total

print("✓ Feature distillation training function defined")

### Run Feature Distillation Training

In [ ]:
# Optimizer and loss
student_feat_optimizer = optim.Adam(student_feat_model.parameters(), lr=1e-3)
feat_distill_criterion = FeatureDistillationLoss(temperature=3.0, alpha=0.2, beta=0.1)

print("Training Student with Feature Distillation...\n")
feat_distill_history = {
    'train_loss': [], 'train_hard_loss': [], 'train_soft_loss': [], 'train_feat_loss': [],
    'train_acc': [], 'test_loss': [], 'test_acc': []
}

for epoch in range(NUM_EPOCHS):
    # Train with feature distillation
    train_loss, hard_loss, soft_loss, feat_loss, train_acc = train_epoch_feature_distillation(
        student_feat_model, teacher, train_loader, feat_distill_criterion,
        student_feat_optimizer, device
    )
    
    # Evaluate (use the base student model for evaluation)
    student_feat_model.eval()
    test_loss, test_acc = evaluate(student_feat, test_loader, criterion, device, "Feat Eval")
    
    feat_distill_history['train_loss'].append(train_loss)
    feat_distill_history['train_hard_loss'].append(hard_loss)
    feat_distill_history['train_soft_loss'].append(soft_loss)
    feat_distill_history['train_feat_loss'].append(feat_loss)
    feat_distill_history['train_acc'].append(train_acc)
    feat_distill_history['test_loss'].append(test_loss)
    feat_distill_history['test_acc'].append(test_acc)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}: Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

print(f"\n✓ Feature distillation training complete!")
print(f"Final test accuracy: {feat_distill_history['test_acc'][-1]:.2f}%")

### Compare All Distillation Methods

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test accuracy comparison
axes[0].plot(teacher_history['test_acc'], label='Teacher (large)', marker='o', linewidth=2.5)
axes[0].plot(vanilla_history['test_acc'], label='Vanilla student', marker='s', linewidth=2)
axes[0].plot(distill_history['test_acc'], label='Response distillation', marker='^', linewidth=2)
axes[0].plot(feat_distill_history['test_acc'], label='Feature distillation', marker='d', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Test Accuracy (%)', fontsize=11)
axes[0].set_title('Distillation Methods Comparison', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Loss components for feature distillation
axes[1].plot(feat_distill_history['train_hard_loss'], label='Hard loss', marker='o')
axes[1].plot(feat_distill_history['train_soft_loss'], label='Soft loss', marker='s')
axes[1].plot(feat_distill_history['train_feat_loss'], label='Feature loss', marker='^')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Loss', fontsize=11)
axes[1].set_title('Feature Distillation Loss Components', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("COMPREHENSIVE COMPARISON")
print("="*70)
print(f"Teacher (large model):         {teacher_history['test_acc'][-1]:.2f}% | {teacher_params:,} params")
print("-" * 70)
print(f"Vanilla student:               {vanilla_history['test_acc'][-1]:.2f}% | {student_params:,} params")
print(f"Response distillation:         {distill_history['test_acc'][-1]:.2f}% | {student_params:,} params")
print(f"Feature distillation:          {feat_distill_history['test_acc'][-1]:.2f}% | {student_params:,} params")
print("="*70)
print(f"\nImprovements over vanilla:")
print(f"  Response distillation: +{distill_history['test_acc'][-1] - vanilla_history['test_acc'][-1]:.2f}%")
print(f"  Feature distillation:  +{feat_distill_history['test_acc'][-1] - vanilla_history['test_acc'][-1]:.2f}%")
print(f"\nCompression: {teacher_params / student_params:.1f}x smaller model")
print("="*70)

## Part 8: Inference Speed Comparison

Knowledge distillation's main benefit is deployment efficiency. Let's measure the actual speedup!

### Measure Inference Time

In [ ]:
def measure_inference_time(model, loader, device, num_batches=50):
    """Measure average inference time per batch."""
    model.eval()
    times = []
    
    with torch.no_grad():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches:
                break
            
            images = images.to(device)
            
            # Warm up (first batch is often slower)
            if i == 0:
                _ = model(images)
                continue
            
            # Time inference
            start = time.time()
            _ = model(images)
            if device.type == 'cuda':
                torch.cuda.synchronize()  # Wait for GPU
            elapsed = time.time() - start
            times.append(elapsed)
    
    return np.mean(times), np.std(times)

print("Measuring inference speed...\n")

# Measure teacher
teacher_time, teacher_std = measure_inference_time(teacher, test_loader, device)
print(f"Teacher: {teacher_time*1000:.2f} ± {teacher_std*1000:.2f} ms/batch")

# Measure student
student_time, student_std = measure_inference_time(student_distill, test_loader, device)
print(f"Student: {student_time*1000:.2f} ± {student_std*1000:.2f} ms/batch")

speedup = teacher_time / student_time
print(f"\nSpeedup: {speedup:.1f}x faster")
print(f"\nFor real-time applications:")
print(f"  Teacher: {1/teacher_time:.1f} batches/second")
print(f"  Student: {1/student_time:.1f} batches/second")

### Visualize Accuracy vs Speed Trade-off

In [ ]:
# Prepare data for visualization
models_data = [
    {'name': 'Teacher', 'acc': teacher_history['test_acc'][-1], 
     'time': teacher_time*1000, 'params': teacher_params},
    {'name': 'Student\n(vanilla)', 'acc': vanilla_history['test_acc'][-1], 
     'time': student_time*1000, 'params': student_params},
    {'name': 'Student\n(distilled)', 'acc': distill_history['test_acc'][-1], 
     'time': student_time*1000, 'params': student_params},
]

# Create scatter plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy vs Inference Time
for data in models_data:
    axes[0].scatter(data['time'], data['acc'], s=data['params']/500, alpha=0.6)
    axes[0].annotate(data['name'], (data['time'], data['acc']), 
                    xytext=(10, 5), textcoords='offset points', fontsize=10)

axes[0].set_xlabel('Inference Time (ms/batch)', fontsize=11)
axes[0].set_ylabel('Test Accuracy (%)', fontsize=11)
axes[0].set_title('Accuracy vs Speed Trade-off', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Bar chart: accuracy comparison
names = [d['name'].replace('\n', ' ') for d in models_data]
accs = [d['acc'] for d in models_data]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
bars = axes[1].bar(names, accs, color=colors, alpha=0.7)
axes[1].set_ylabel('Test Accuracy (%)', fontsize=11)
axes[1].set_title('Final Accuracy Comparison', fontsize=12, fontweight='bold')
axes[1].set_ylim([min(accs)-5, max(accs)+2])

# Add value labels on bars
for bar, acc in zip(bars, accs):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{acc:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("\nKnowledge distillation achieves the sweet spot:")
print(f"  ✓ Student speed ({student_time*1000:.1f}ms)")
print(f"  ✓ Near-teacher accuracy ({distill_history['test_acc'][-1]:.1f}%)")
print(f"  ✓ Small model size ({student_params:,} params)")

## Part 9: Practical Deployment Considerations

### When to Use Knowledge Distillation

Knowledge distillation is ideal when:

**✅ Use distillation when:**
- Deploying to resource-constrained devices (mobile, edge, IoT)
- Need low-latency inference (real-time applications)
- Want to reduce cloud costs (smaller models = cheaper hosting)
- Have a strong teacher model already trained
- Can afford small accuracy drop (1-3%) for large speedup

**❌ Don't use distillation when:**
- Maximum accuracy is critical (use full ensemble instead)
- Inference speed is not a constraint
- Teacher is not significantly better than student
- Don't have resources to train teacher first

### Hyperparameter Selection Guide

**Temperature ($T$)**:
- Typical range: 2-5
- Too low (T=1): Soft targets too sharp, similar to hard labels
- Too high (T>10): Soft targets too uniform, loses information
- Start with T=3, adjust based on validation performance

**Alpha ($\alpha$) - Hard loss weight**:
- Typical range: 0.1-0.3
- Higher α: More weight on ground truth (conservative)
- Lower α: More weight on teacher (trust teacher more)
- Start with α=0.2

**Beta ($\beta$) - Feature loss weight** (for feature distillation):
- Typical range: 0.05-0.2
- Balance between output matching and feature matching
- Start with β=0.1

### Teacher-Student Architecture Selection

**Teacher model:**
- Use your best, most accurate model
- Ensemble of models can be a teacher
- Larger models produce better soft targets

**Student model:**
- Should have similar architecture "family" as teacher (e.g., both CNNs)
- Compression ratio: typically 2-10x smaller
- Too small: Can't capture teacher's knowledge
- Too large: Defeats the purpose of compression

**Rule of thumb**: Student should have 10-30% of teacher's parameters.

### Advanced Techniques

**1. Self-distillation**: Use the same architecture for teacher and student
- Teacher trained first, then distills into identical student
- Student often matches or exceeds teacher (regularization effect)

**2. Multi-teacher distillation**: Student learns from multiple teachers
- Average soft targets from multiple models
- Captures diverse knowledge

**3. Online distillation**: Teacher and student train simultaneously
- More efficient (no separate teacher training)
- Mutual learning between models

**4. Progressive distillation**: Chain of student→teacher→student
- First student becomes teacher for second student
- Gradually compress model over multiple stages

### Real-World Applications

Knowledge distillation is widely used in industry:

**Computer Vision:**
- Mobile apps (e.g., Google Lens, iPhone Face ID)
- Autonomous vehicles (real-time object detection)
- Video surveillance (continuous processing)

**Natural Language Processing:**
- DistilBERT: 97% of BERT accuracy, 2x faster, 40% smaller
- TinyBERT: 96% accuracy, 7.5x smaller, 9.4x faster
- Mobile keyboards (autocomplete, prediction)

**Speech Recognition:**
- On-device voice assistants
- Real-time transcription

**Recommendation Systems:**
- Low-latency serving at scale
- Edge deployment for personalization

### Deployment Workflow

**Step 1: Train Teacher**
- Use all available tricks: ensembles, large models, heavy augmentation
- Optimize for maximum accuracy (speed doesn't matter)
- Take as long as needed

**Step 2: Design Student**
- Target your deployment constraints (latency, memory, power)
- Start with 5-10x compression, adjust based on results
- Profile on target hardware

**Step 3: Distillation**
- Try response-based first (simpler, often sufficient)
- If needed, add feature distillation
- Tune temperature and loss weights on validation set

**Step 4: Validation**
- Compare accuracy drop (should be <3%)
- Measure actual speedup on target device
- Test edge cases (adversarial examples, distribution shift)

**Step 5: Deploy**
- Export student model (ONNX, TensorFlow Lite, etc.)
- Quantize if needed (INT8, further 4x speedup)
- Monitor performance in production

## Key Takeaways

### Core Concepts

1. **Knowledge distillation** teaches small models to mimic large models, achieving similar accuracy with much better efficiency

2. **Soft targets** from teacher models are much richer than hard labels:
   - Encode class relationships
   - Provide smoother gradients
   - Transfer "dark knowledge" about the task

3. **Temperature scaling** controls the "softness" of predictions:
   - T=1: Normal softmax (sharp)
   - T=3-5: Good for distillation (reveals relationships)
   - T>10: Too soft (loses information)

4. **Response-based distillation** matches output probabilities:
   - Simple and effective
   - Works well for most applications
   - Combines hard labels and soft targets

5. **Feature-based distillation** matches intermediate representations:
   - More complex but often better results
   - Student learns similar features to teacher
   - Useful for complex tasks

### Practical Guidelines

**Model selection:**
- Teacher: Use your best, most accurate model
- Student: 10-30% of teacher's parameters
- Similar architectures work best

**Hyperparameters:**
- Temperature: Start with T=3
- Hard loss weight: α=0.2 (20% hard, 80% soft)
- Feature loss weight: β=0.1 (if using features)

**Expected results:**
- 5-10x smaller models
- 5-20x faster inference
- 1-3% accuracy drop

### When to Use

**Use knowledge distillation when:**
- ✅ Deploying to mobile/edge devices
- ✅ Need real-time inference
- ✅ Want to reduce serving costs
- ✅ Have a strong teacher model

**Don't use when:**
- ❌ Maximum accuracy is critical
- ❌ Inference speed not a concern
- ❌ Don't have good teacher yet

### The Big Picture

Knowledge distillation bridges the gap between **research models** (large, accurate, slow) and **production models** (small, fast, deployable).

It's one of the most practical techniques in deep learning - used in nearly every production ML system that needs to serve predictions efficiently at scale.

**Key insight**: The teacher's mistakes and uncertainties contain valuable information! By learning to mimic the teacher's full probability distribution (not just its final predictions), students can approach the teacher's accuracy with a fraction of the parameters.

## Further Exploration

Try these experiments to deepen your understanding:

1. **Temperature sensitivity**: Try T=1, 2, 3, 5, 10, 20
   - How does student performance change?
   - Visualize soft targets at different temperatures

2. **Loss weight ablation**: Try α=0, 0.1, 0.5, 1.0
   - What happens with only soft loss (α=0)?
   - What about only hard loss (α=1)?

3. **Compression ratio**: Try students of different sizes
   - 2x, 5x, 10x, 20x smaller than teacher
   - Find the sweet spot for your application

4. **Self-distillation**: Use same architecture for teacher and student
   - Does student match or exceed teacher?

5. **Quantization + distillation**: Combine with INT8 quantization
   - How much total speedup can you achieve?

6. **Different datasets**: Try on Fashion-MNIST, SVHN, or your own data
   - Does distillation work better on some tasks?

Happy experimenting! 🚀